In [ ]:
# In[1]:

import pandas as pd

# Load CSVs
metric_df = pd.read_csv("metric.csv")
trace_df = pd.read_csv("trace.csv")
log_df = pd.read_csv("log.csv")
error_df = pd.read_csv("error_logs.csv")

# Parse timestamps to UTC datetimes (Unix seconds -> UTC)
for df in (metric_df, trace_df, log_df, error_df):
    df['timestamp'] = pd.to_datetime(df['timestamp'], unit='s', utc=True)

# Helper to format timestamps as ISO UTC strings
def to_iso_utc(ts):
    # ts may be a Timestamp or Series of Timestamps
    if isinstance(ts, pd.Series):
        return ts.dt.strftime('%Y-%m-%dT%H:%M:%SZ')
    else:
        return pd.to_datetime(ts).strftime('%Y-%m-%dT%H:%M:%SZ')

# 1) metric.csv summary: group by (cmdb_id, kpi_name)
metric_grouped = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'])['value']
    .agg(count='count',
         earliest_ts=lambda x: metric_df.loc[x.index, 'timestamp'].min(),
         latest_ts=lambda x: metric_df.loc[x.index, 'timestamp'].max(),
         p95=lambda x: x.quantile(0.95),
         p90=lambda x: x.quantile(0.90))
    .reset_index()
)

# Format timestamps to ISO UTC strings
metric_grouped['earliest'] = to_iso_utc(metric_grouped['earliest_ts'])
metric_grouped['latest'] = to_iso_utc(metric_grouped['latest_ts'])
metric_summary = metric_grouped[['cmdb_id', 'kpi_name', 'count', 'earliest', 'latest', 'p95', 'p90']].sort_values('count', ascending=False).head(50)

# 2) trace.csv summary: group by (cmdb_id, trace_name)
trace_grouped = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'])['value']
    .agg(count='count',
         earliest_ts=lambda x: trace_df.loc[x.index, 'timestamp'].min(),
         latest_ts=lambda x: trace_df.loc[x.index, 'timestamp'].max(),
         p95=lambda x: x.quantile(0.95),
         p90=lambda x: x.quantile(0.90))
    .reset_index()
)

trace_grouped['earliest'] = to_iso_utc(trace_grouped['earliest_ts'])
trace_grouped['latest'] = to_iso_utc(trace_grouped['latest_ts'])
trace_summary = trace_grouped[['cmdb_id', 'trace_name', 'count', 'earliest', 'latest', 'p95', 'p90']].sort_values('count', ascending=False).head(50)

# 3) log.csv summary: group by (cmdb_id, log_name)
log_grouped = (
    log_df
    .groupby(['cmdb_id', 'log_name'])['value']
    .agg(count='count',
         earliest_ts=lambda x: log_df.loc[x.index, 'timestamp'].min(),
         latest_ts=lambda x: log_df.loc[x.index, 'timestamp'].max(),
         p95=lambda x: x.quantile(0.95),
         p90=lambda x: x.quantile(0.90))
    .reset_index()
)

log_grouped['earliest'] = to_iso_utc(log_grouped['earliest_ts'])
log_grouped['latest'] = to_iso_utc(log_grouped['latest_ts'])
log_summary = log_grouped[['cmdb_id', 'log_name', 'count', 'earliest', 'latest', 'p95', 'p90']].sort_values('count', ascending=False).head(50)

# 4) error_logs.csv summary
error_total_count = len(error_df)
error_per_cmdb = error_df.groupby('cmdb_id').size().reset_index(name='count').sort_values('count', ascending=False)
error_earliest = error_df['timestamp'].min() if error_total_count>0 else pd.NaT
error_latest = error_df['timestamp'].max() if error_total_count>0 else pd.NaT
error_overall = pd.DataFrame([{
    'total_count': error_total_count,
    'earliest': to_iso_utc(error_earliest) if pd.notna(error_earliest) else None,
    'latest': to_iso_utc(error_latest) if pd.notna(error_latest) else None
}])

# Sample up to 5 messages (timestamp, cmdb_id, message) sorted by timestamp ascending
error_samples = error_df.sort_values('timestamp', ascending=True).head(5).copy()
if not error_samples.empty:
    error_samples['timestamp'] = to_iso_utc(error_samples['timestamp'])

# Display compact summaries (variables returned)
metric_summary, trace_summary, log_summary, error_per_cmdb.head(50), error_overall, error_samples[['timestamp','cmdb_id','message']].head(5)

```
Out[1]:
```
```python
# Build a concise plain-English summary using the previously computed summaries in the IPython state.

# Reuse existing DataFrame summaries: metric_summary, trace_summary, log_summary, error_overall
# (They were created in the prior cell.)

# Helper to format small tables into text lines
def rows_to_lines(df, cols, limit=3):
    lines = []
    for _, r in df[cols].head(limit].iterrows():
        parts = [f"{c}: {r[c]}" for c in cols]
        lines.append(" | ".join(parts))
    return lines

# Prepare top-3 rows for each category (compact)
metric_top3 = metric_summary[['cmdb_id','kpi_name','count','p95','p90']].head(3)
trace_top3  = trace_summary[['cmdb_id','trace_name','count','p95','p90']].head(3)
log_top3    = log_summary[['cmdb_id','log_name','count','p95','p90']].head(3)

metric_lines = [f"{r['cmdb_id']} - {r['kpi_name']} (count={int(r['count'])}, p95={r['p95']}, p90={r['p90']})"
                for _, r in metric_top3.iterrows()]
trace_lines  = [f"{r['cmdb_id']} - {r['trace_name']} (count={int(r['count'])}, p95={r['p95']}, p90={r['p90']})"
                for _, r in trace_top3.iterrows()]
log_lines    = [f"{r['cmdb_id']} - {r['log_name']} (count={int(r['count'])}, p95={r['p95']}, p90={r['p90']})"
                for _, r in log_top3.iterrows()]

# Determine overall telemetry window from metric groups (they cover the observed window)
time_start = metric_grouped['earliest_ts'].min()
time_end   = metric_grouped['latest_ts'].max()
time_start_iso = pd.to_datetime(time_start).strftime('%Y-%m-%dT%H:%M:%SZ') if pd.notna(time_start) else "N/A"
time_end_iso   = pd.to_datetime(time_end).strftime('%Y-%m-%dT%H:%M:%SZ') if pd.notna(time_end) else "N/A"

# Error logs summary
err_total = int(error_overall.at[0, 'total_count'])
err_earliest = error_overall.at[0, 'earliest']
err_latest = error_overall.at[0, 'latest']

summary = (
    f"Telemetry summary (plain English):\n"
    f"- Time window (UTC): {time_start_iso}  to  {time_end_iso}\n"
    f"- Metrics: many metric groups present (most groups have 25 samples across the window).\n"
    f"  Top metric groups (up to 3 shown):\n    " + "\n    ".join(metric_lines) + "\n"
    f"- Traces: traces present for many services (25 samples each in shown groups).\n"
    f"  Top trace groups (up to 3 shown):\n    " + "\n    ".join(trace_lines) + "\n"
    f"- Logs: per-service log metrics present; top examples (up to 3 shown):\n    " + "\n    ".join(log_lines) + "\n"
    f"- Error logs: total_count = {err_total}. Earliest = {err_earliest}, Latest = {err_latest}.\n"
    f"  (No error log messages were recorded in error_logs.csv for this window.)\n\n"
    f"Overall conclusion: telemetry contains rich metric/trace/log coverage across the window "
    f"{time_start_iso}–{time_end_iso}. Log error_count values in top groups are zero, and error_logs.csv "
    f"contains no entries, so there are no recorded error-log traces in this dataset for the analyzed interval."
)

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

(                         cmdb_id kpi_name  count              earliest                latest           p95           p90
359           ts-voucher-service   socket     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  4.000000e+00  4.000000e+00
358           ts-voucher-service      mem     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  3.805960e+07  3.801498e+07
357           ts-voucher-service   diskio     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  0.000000e+00  0.000000e+00
2    ts-admin-basic-info-service      cpu     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  2.767540e+00  2.180443e+00
3    ts-admin-basic-info-service   diskio     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  6.612386e+04  6.257724e+04
..                           ...      ...    ...                   ...                   ...           ...           ...
69               ts-config-mongo      mem     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  6.741096e+07  6.740484e+07
54             ts-avatar-service      mem     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  1.341545e+08  1.341543e+08
55             ts-avatar-service   socket     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  5.000000e+00  5.000000e+00
56              ts-basic-service      cpu     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  1.943868e+01  1.779986e+01
57              ts-basic-service   diskio     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  6.220613e+04  5.927201e+04

[50 rows x 7 columns],                 cmdb_id                                     trace_name  count              earliest                latest        p95        p90
728  ts-travel2-service  trace.to_ts-order-other-service.duration_mean     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.013669   0.010774
729  ts-travel2-service   trace.to_ts-order-other-service.duration_p95     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.070409   0.051075
730  ts-travel2-service     trace.to_ts-order-other-service.error_rate     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.000000   0.000000
731  ts-travel2-service      trace.to_ts-order-other-service.row_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  92.600000  90.600000
732  ts-travel2-service        trace.to_ts-route-service.duration_mean     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.006505   0.006323
..                  ...                                            ...    ...                   ...                   ...        ...        ...
45                 root      trace.to_ts-preserve-service.duration_p95     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   1.602463   1.526849
46                 root        trace.to_ts-preserve-service.error_rate     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.000000   0.000000
47                 root         trace.to_ts-preserve-service.row_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   7.000000   7.000000
32                 root  trace.to_ts-order-other-service.duration_mean     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.107115   0.101653
33                 root   trace.to_ts-order-other-service.duration_p95     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.141877   0.140319

[50 rows x 7 columns],                         cmdb_id         log_name  count              earliest                latest   p95   p90
0   ts-admin-basic-info-service  log.error_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.0   0.0
1   ts-admin-basic-info-service    log.row_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  32.8  32.0
2       ts-admin-travel-service  log.error_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.0   0.0
3       ts-admin-travel-service    log.row_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  32.8  31.2
7          ts-assurance-service    log.row_count     25  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z  15.4  12.6
..                          ...              ...    ...                   ...                   ...   ...   ...
18       ts-consign-price-mongo  log.error_count     24  2024-02-20T18:06:00Z  2024-02-20T18:29:00Z   0.0   0.0
14              ts-config-mongo  log.error_count     24  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   0.0   0.0
15              ts-config-mongo    log.row_count     24  2024-02-20T18:05:00Z  2024-02-20T18:29:00Z   1.0   1.0
16             ts-consign-mongo  log.error_count     24  2024-02-20T18:06:00Z  2024-02-20T18:29:00Z   0.0   0.0
17             ts-consign-mongo    log.row_count     24  2024-02-20T18:06:00Z  2024-02-20T18:29:00Z   1.0   1.0

[50 rows x 7 columns], Empty DataFrame
Columns: [cmdb_id, count]
Index: [],    total_count earliest latest
0            0     None   None, Empty DataFrame
Columns: [timestamp, cmdb_id, message]
Index: [])

**Note**: The printed pandas DataFrame is truncated due to its size. Only **10 rows** are displayed, which may introduce observation bias due to the incomplete table. If you want to comprehensively understand the details without bias, please ask Executor using `df.head(X)` to display more rows.```
```

In [ ]:
# In[2]:

import pandas as pd
import numpy as np

# Ensure metric_df is loaded (reuse if present)
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
    metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp dtype correct
    if metric_df['timestamp'].dtype == 'int64' or metric_df['timestamp'].dtype == 'float64':
        metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# 1) Compute global P95 per (cmdb_id, kpi_name) using all data
p95_df = (
    metric_df
    .groupby(['cmdb_id', 'kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)

# Merge p95 back into metric_df for easy comparison
metric_with_p95 = metric_df.merge(p95_df, on=['cmdb_id', 'kpi_name'], how='left')

# 2) Filter to incident window (inclusive)
start = pd.to_datetime("2024-02-20 18:02:36", utc=True)
end   = pd.to_datetime("2024-02-20 18:32:36", utc=True)
metric_window = metric_with_p95[(metric_with_p95['timestamp'] >= start) & (metric_with_p95['timestamp'] <= end)].copy()

# 3) For each group in the window, find anomaly rows where value > p95 and identify consecutive segments
rows = []
grouped = metric_window.groupby(['cmdb_id', 'kpi_name'], sort=False)

for (cmdb_id, kpi_name), g in grouped:
    g = g.sort_values('timestamp').reset_index(drop=True)
    p95_value = g['p95'].iat[0] if not g['p95'].isna().all() else np.nan
    # Mark anomalies (value > p95). If p95 is NaN, no anomalies.
    if pd.isna(p95_value):
        continue
    anomalies = g[g['value'] > p95_value].copy()
    if anomalies.empty:
        continue
    # Identify consecutive segments among anomaly rows (consecutive if timestamp diff == 60s)
    anomalies = anomalies.sort_values('timestamp').reset_index(drop=True)
    dt = anomalies['timestamp'].diff().dt.total_seconds().fillna(99999)
    # new segment when dt != 60
    seg_id = (dt != 60).cumsum()
    anomalies['seg_id'] = seg_id
    seg_sizes = anomalies.groupby('seg_id').size()
    longest_consec = int(seg_sizes.max())
    # Only keep groups with a consecutive anomaly segment length >= 2
    if longest_consec < 2:
        continue
    count_anomalous_points = int(len(anomalies))
    earliest_anom = anomalies['timestamp'].min()
    latest_anom = anomalies['timestamp'].max()
    # Max value in the window for this (cmdb_id,kpi_name) (use all points in the window, not just anomalies)
    max_value_in_window = float(g['value'].max())
    # max_over_p95: use max_value_in_window / p95_value, Inf if p95==0
    if p95_value == 0:
        max_over_p95 = np.inf
    else:
        max_over_p95 = float(max_value_in_window / p95_value)
    # sample up to 5 anomaly timestamps (ISO UTC strings)
    sample_ts = anomalies['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ').head(5).tolist()
    earliest_iso = earliest_anom.strftime('%Y-%m-%dT%H:%M:%SZ')
    latest_iso = latest_anom.strftime('%Y-%m-%dT%H:%M:%SZ')
    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'p95': float(p95_value),
        'count_anomalous_points': count_anomalous_points,
        'longest_consecutive_length': longest_consec,
        'earliest_anomaly_timestamp': earliest_iso,
        'latest_anomaly_timestamp': latest_iso,
        'max_value_in_window': max_value_in_window,
        'max_over_p95': max_over_p95,
        'sample_anomaly_timestamps': sample_ts
    })

# 4) Build result DataFrame, sort and limit to top 20 by max_over_p95 desc
result_df = pd.DataFrame(rows)
if not result_df.empty:
    # replace inf with a large sentinel for sorting so Inf come first; we'll keep Inf as-is in output
    result_df['_sort_key'] = result_df['max_over_p95'].replace(np.inf, np.finfo('float64').max)
    result_df = result_df.sort_values('_sort_key', ascending=False).drop(columns='_sort_key')
    # Ensure proper column order
    cols = ['cmdb_id','kpi_name','p95','count_anomalous_points','longest_consecutive_length',
            'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','max_over_p95','sample_anomaly_timestamps']
    result_df = result_df[cols].head(20)

result_df

```
Out[2]:
```
```python
# Summarize the anomaly detection results in plain English using the existing result_df from the IPython state.

if 'result_df' not in globals() or result_df is None or result_df.empty:
    summary = "No candidate anomaly groups (with consecutive anomaly length >=2) were found in the incident window."
else:
    total_candidates = len(result_df)
    # Top 3 by severity (max_over_p95)
    top_n = result_df.head(3)
    top_lines = []
    for i, r in top_n.iterrows():
        top_lines.append(
            f"- {r['cmdb_id']} | {r['kpi_name']} — p95={r['p95']:.6f}, max={r['max_value_in_window']:.6f}, "
            f"max_over_p95={r['max_over_p95']:.3f}, anomalies={r['count_anomalous_points']}, "
            f"longest_seq={r['longest_consecutive_length']}, window={r['earliest_anomaly_timestamp']} to {r['latest_anomaly_timestamp']}"
        )
    # General observations
    # Most groups show two consecutive anomalous minutes around 18:05–18:06 UTC
    earliest_overall = result_df['earliest_anomaly_timestamp'].min()
    latest_overall = result_df['latest_anomaly_timestamp'].max()
    common_counts = result_df['count_anomalous_points'].value_counts().to_dict()
    most_common_count = sorted(common_counts.items(), key=lambda x: -x[1])[0]
    summary = (
        f"Anomaly detection summary for 2024-02-20 18:02:36–18:32:36 UTC:\n\n"
        f"Total candidate groups with consecutive anomaly segment >=2: {total_candidates}.\n\n"
        f"Top candidates by severity (max_over_p95):\n" + "\n".join(top_lines) + "\n\n"
        f"Observations:\n"
        f"- Anomalies are tightly clustered in time: overall earliest anomaly = {earliest_overall}, "
        f"latest anomaly = {latest_overall}.\n"
        f"- Most candidate groups show {most_common_count[0]} anomalous points (occurs for {most_common_count[1]} groups), "
        f"and the typical longest consecutive anomaly length is 2 minutes (i.e., two consecutive per-minute points).\n"
        f"- Example top severity: ts-security-service / latency-90 had p95={result_df.iloc[0]['p95']:.6f} and peaked at "
        f"{result_df.iloc[0]['max_value_in_window']:.6f} (max_over_p95={result_df.iloc[0]['max_over_p95']:.3f}).\n\n"
        f"Conclusion:\n"
        f"- Multiple services experienced short, simultaneous latency spikes (typically 2 consecutive minutes) centered "
        f"around ~2024-02-20T18:05:00Z–2024-02-20T18:06:00Z. The most severe spike in this set is on ts-security-service (latency-90).\n"
        f"- These results indicate a brief, system-wide latency disturbance affecting many services during the incident window."
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name       p95  count_anomalous_points  longest_consecutive_length earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  max_over_p95                     sample_anomaly_timestamps
100          ts-security-service  latency-90  0.235836                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             3.891745     16.501879  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
134              ts-user-service  latency-90  0.224597                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             3.281998     14.612813  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
99           ts-security-service  latency-50  0.091856                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.828975      9.024706  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
51               ts-food-service  latency-90  0.297200                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             2.163478      7.279543  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
81           ts-preserve-service  latency-90  3.793991                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z            19.442610      5.124580  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
39           ts-contacts-service  latency-50  0.020433                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.095208      4.659615  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
28             ts-config-service  latency-90  0.018142                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.074865      4.126570  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
110        ts-ticketinfo-service  latency-50  0.026987                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.094967      3.519052  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
123           ts-travel2-service  latency-50  0.034891                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.113014      3.239012  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
74            ts-payment-service  latency-50  0.042057                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.132078      3.140460  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
20              ts-basic-service  latency-50  0.021340                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.065311      3.060494  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
21              ts-basic-service  latency-90  0.095218                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.265581      2.789172  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
138    ts-preserve-other-service  latency-90  5.107568                       2                           2       2024-02-20T18:06:00Z     2024-02-20T18:07:00Z            12.930266      2.531590  [2024-02-20T18:06:00Z, 2024-02-20T18:07:00Z]
111        ts-ticketinfo-service  latency-90  0.139690                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.349786      2.504026  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
95               ts-seat-service  latency-90  0.316278                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.753615      2.382759  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
117            ts-travel-service  latency-50  0.066704                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.157962      2.368092  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
113             ts-train-service  latency-50  0.009223                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.021734      2.356415  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
94               ts-seat-service  latency-50  0.146318                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.340975      2.330371  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
1    ts-admin-basic-info-service  latency-50  0.026382                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             0.060256      2.283959  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
124           ts-travel2-service  latency-90  0.900021                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z             1.995979      2.217702  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]```
```

In [ ]:
# In[3]:

import pandas as pd
import numpy as np

# Reuse existing trace_df if present, otherwise load and parse timestamps
if 'trace_df' not in globals():
    trace_df = pd.read_csv("trace.csv")
    trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
else:
    # ensure timestamp parsed
    if trace_df['timestamp'].dtype.kind in ('i','u','f'):
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], unit='s', utc=True)
    else:
        # ensure tz-aware
        trace_df['timestamp'] = pd.to_datetime(trace_df['timestamp'], utc=True)

# 1) Compute global p95 per (cmdb_id, trace_name) using full trace.csv
p95_trace = (
    trace_df
    .groupby(['cmdb_id', 'trace_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value': 'p95'})
)

# 2) Filter trace.csv to incident window (inclusive)
start = pd.to_datetime("2024-02-20 18:02:36", utc=True)
end   = pd.to_datetime("2024-02-20 18:32:36", utc=True)
trace_with_p95 = trace_df.merge(p95_trace, on=['cmdb_id','trace_name'], how='left')
trace_window = trace_with_p95[(trace_with_p95['timestamp'] >= start) & (trace_with_p95['timestamp'] <= end)].copy()

# 3) Restrict to outgoing calls (trace_name starts with 'trace.to_') and mark anomalies value > p95
mask_outgoing = trace_window['trace_name'].str.startswith('trace.to_')
trace_window_out = trace_window[mask_outgoing].copy()

# 5) Keep only trace metrics of interest (suffixes)
suffixes = ('.duration_mean', '.duration_p95', '.error_rate')
mask_suffix = trace_window_out['trace_name'].str.endswith(suffixes)
trace_window_out = trace_window_out[mask_suffix].copy()

# Ensure p95 present; mark anomalous rows where value > p95
trace_window_out['is_anom'] = False
# only mark where p95 is finite (not NaN)
valid_p95 = trace_window_out['p95'].notna()
trace_window_out.loc[valid_p95, 'is_anom'] = trace_window_out.loc[valid_p95, 'value'] > trace_window_out.loc[valid_p95, 'p95']

# Keep only anomalous rows
anoms = trace_window_out[trace_window_out['is_anom']].copy()
if not anoms.empty:
    # 4) Extract destination service from trace_name between 'trace.to_' and next '.'
    def extract_dest(s):
        try:
            return s.split('trace.to_',1)[1].split('.',1)[0]
        except Exception:
            return ""
    anoms['destination_service'] = anoms['trace_name'].map(extract_dest)

    # compute ratio value/p95, handle p95==0 => inf
    anoms['ratio_over_p95'] = np.where(anoms['p95'] == 0, np.inf, anoms['value'] / anoms['p95'])

    # 6) Group anomalous rows by destination service and compute aggregates
    grouped = anoms.groupby('destination_service', sort=False)

    rows = []
    for dest, g in grouped:
        distinct_callers = int(g['cmdb_id'].nunique())
        total_points = int(len(g))
        earliest_iso = g['timestamp'].min().strftime('%Y-%m-%dT%H:%M:%SZ')
        latest_iso = g['timestamp'].max().strftime('%Y-%m-%dT%H:%M:%SZ')
        max_over_p95 = float(np.inf) if np.isinf(g['ratio_over_p95']).any() else float(g['ratio_over_p95'].max())
        sample_callers = sorted(g['cmdb_id'].unique())[:5]
        rows.append({
            'destination_service': dest,
            'distinct_callers_with_anomalies': distinct_callers,
            'total_anomalous_points': total_points,
            'earliest_anomaly_timestamp': earliest_iso,
            'latest_anomaly_timestamp': latest_iso,
            'max_over_p95': max_over_p95,
            'sample_callers': sample_callers
        })

    dest_summary = pd.DataFrame(rows)
    # 7) Sort by distinct_callers desc, then max_over_p95 desc, limit top 10
    dest_summary['_sort_key_inf'] = dest_summary['max_over_p95'].replace(np.inf, np.finfo('float64').max)
    dest_summary = dest_summary.sort_values(by=['distinct_callers_with_anomalies','_sort_key_inf'], ascending=[False, False]).drop(columns=['_sort_key_inf'])
    dest_summary_top10 = dest_summary.head(10)
else:
    # empty result structure
    dest_summary_top10 = pd.DataFrame(columns=[
        'destination_service','distinct_callers_with_anomalies','total_anomalous_points',
        'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_over_p95','sample_callers'
    ])

# Display the compact result table (top 10)
dest_summary_top10

```
Out[3]:
```
```python
summary = (
    "Trace anomaly summary for 2024-02-20 18:02:36–18:32:36 UTC:\n\n"
    "Top affected downstream services (top 10):\n"
    "1) ts-order-other-service — 7 distinct callers, 28 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:29:00Z, max_over_p95 ≈ 6.61\n"
    "2) ts-station-service — 7 distinct callers, 28 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:29:00Z, max_over_p95 ≈ 1.65\n"
    "3) ts-order-service — 7 distinct callers, 28 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:29:00Z, max_over_p95 ≈ 1.41\n"
    "4) ts-travel-service — 6 distinct callers, 24 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:25:00Z, max_over_p95 ≈ 1.27\n"
    "5) ts-seat-service — 5 distinct callers, 20 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:26:00Z, max_over_p95 ≈ 2.06\n"
    "6) ts-ticketinfo-service — 5 distinct callers, 20 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:09:00Z, max_over_p95 ≈ 2.00\n"
    "7) ts-travel2-service — 5 distinct callers, 20 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:09:00Z, max_over_p95 ≈ 1.63\n"
    "8) ts-assurance-service — 4 distinct callers, 14 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:14:00Z, max_over_p95 ≈ 11.73 (highest ratio)\n"
    "9) ts-contacts-service — 4 distinct callers, 16 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:29:00Z, max_over_p95 ≈ 4.29\n"
    "10) ts-train-service — 4 distinct callers, 16 anomalous points, anomalies 2024-02-20T18:05:00Z to 2024-02-20T18:12:00Z, max_over_p95 ≈ 2.64\n\n"
    "Key observations:\n"
    "- Anomalies begin around 2024-02-20T18:05:00Z and many persist across the window (up to 18:29Z for several services).\n"
    "- Order-related endpoints (ts-order-other-service, ts-order-service) show the widest caller impact (7 distinct callers each) and strong severity (ts-order-other-service max_over_p95 ≈ 6.6).\n"
    "- ts-assurance-service shows the largest ratio over its global p95 (≈11.7) though with fewer callers — this indicates very large deviations where it occurs.\n"
    "- The pattern (many callers hitting the same downstream targets) suggests the issue is concentrated at those downstream services rather than isolated callers.\n\n"
    "Recommendation: prioritize investigation of ts-order-other-service and ts-assurance-service (and nearby order/station/travel downstreams) for root-cause during the incident window."
)
summary
```

The original code execution output of IPython Kernel is also provided below for reference:

destination_service  distinct_callers_with_anomalies  total_anomalous_points earliest_anomaly_timestamp latest_anomaly_timestamp  max_over_p95                                     sample_callers
8   ts-order-other-service                                7                      28       2024-02-20T18:05:00Z     2024-02-20T18:29:00Z      6.605948  [root, ts-inside-payment-service, ts-order-oth...
15      ts-station-service                                7                      28       2024-02-20T18:05:00Z     2024-02-20T18:29:00Z      1.649640  [ts-basic-service, ts-food-service, ts-order-o...
19        ts-order-service                                7                      28       2024-02-20T18:05:00Z     2024-02-20T18:29:00Z      1.408778  [root, ts-inside-payment-service, ts-order-ser...
10       ts-travel-service                                6                      24       2024-02-20T18:05:00Z     2024-02-20T18:25:00Z      1.266021  [root, ts-admin-travel-service, ts-food-servic...
20         ts-seat-service                                5                      20       2024-02-20T18:05:00Z     2024-02-20T18:26:00Z      2.056357  [ts-preserve-other-service, ts-preserve-servic...
24   ts-ticketinfo-service                                5                      20       2024-02-20T18:05:00Z     2024-02-20T18:09:00Z      2.001658  [ts-preserve-other-service, ts-preserve-servic...
11      ts-travel2-service                                5                      20       2024-02-20T18:05:00Z     2024-02-20T18:09:00Z      1.634502  [root, ts-admin-travel-service, ts-preserve-ot...
2     ts-assurance-service                                4                      14       2024-02-20T18:05:00Z     2024-02-20T18:14:00Z     11.730509  [root, ts-assurance-service, ts-preserve-other...
5      ts-contacts-service                                4                      16       2024-02-20T18:05:00Z     2024-02-20T18:29:00Z      4.287203  [root, ts-contacts-service, ts-preserve-other-...
16        ts-train-service                                4                      16       2024-02-20T18:05:00Z     2024-02-20T18:12:00Z      2.636205  [ts-basic-service, ts-train-service, ts-travel...```
```

In [ ]:
# In[4]:

import pandas as pd
import numpy as np

# Reuse metric_df if present, otherwise load and parse timestamps
if 'metric_df' not in globals():
    metric_df = pd.read_csv("metric.csv")
metric_df['timestamp'] = pd.to_datetime(metric_df['timestamp'], unit='s', utc=True)

# Components of interest
components = ['ts-auth-service','ts-order-service','ts-route-service','ts-train-service','ts-travel-service']

# 1) Compute global P95 per (cmdb_id, kpi_name) using the full metric_df
p95_metric = (
    metric_df
    .groupby(['cmdb_id','kpi_name'], as_index=False)['value']
    .quantile(0.95)
    .rename(columns={'value':'p95'})
)

# 2) Filter metric_df to incident window (inclusive)
start = pd.to_datetime("2024-02-20 18:02:36", utc=True)
end   = pd.to_datetime("2024-02-20 18:32:36", utc=True)
metric_window = metric_df[(metric_df['timestamp'] >= start) & (metric_df['timestamp'] <= end)].copy()

# Further restrict to the specified components
metric_window = metric_window[metric_window['cmdb_id'].isin(components)].copy()

# Merge p95 (global) into the windowed data
metric_window = metric_window.merge(p95_metric, on=['cmdb_id','kpi_name'], how='left')

# 3) For each group in the window, mark anomalies value > p95 and find consecutive segments
rows = []
grouped = metric_window.groupby(['cmdb_id','kpi_name'], sort=False)

for (cmdb_id, kpi_name), g in grouped:
    g = g.sort_values('timestamp').reset_index(drop=True)
    # If p95 is missing (NaN) for this group, skip
    if g['p95'].isna().all():
        continue
    p95_val = float(g['p95'].iat[0])
    # Mark anomalies where value > p95
    anom_mask = g['value'] > p95_val
    anomalies = g[anom_mask].copy()
    if anomalies.empty:
        continue
    # Identify consecutive segments among anomalies (consecutive if timestamp diff == 60s)
    anomalies = anomalies.sort_values('timestamp').reset_index(drop=True)
    dt = anomalies['timestamp'].diff().dt.total_seconds().fillna(99999)
    seg_id = (dt != 60).cumsum()  # new segment when gap != 60
    anomalies['seg_id'] = seg_id
    seg_sizes = anomalies.groupby('seg_id').size()
    longest_consec = int(seg_sizes.max())
    # 4) Keep only groups with longest_consec >= 2
    if longest_consec < 2:
        continue
    count_anomalous_points = int(len(anomalies))
    earliest_anom = anomalies['timestamp'].min()
    latest_anom = anomalies['timestamp'].max()
    # max value in the window for this group (use all points in the window)
    max_value_in_window = float(g['value'].max())
    # max_over_p95: handle p95==0
    if p95_val == 0:
        max_over_p95 = np.inf
    else:
        max_over_p95 = float(max_value_in_window / p95_val)
    # sample up to 5 anomaly timestamps
    sample_ts = anomalies['timestamp'].dt.strftime('%Y-%m-%dT%H:%M:%SZ').head(5).tolist()
    rows.append({
        'cmdb_id': cmdb_id,
        'kpi_name': kpi_name,
        'p95': p95_val,
        'count_anomalous_points': count_anomalous_points,
        'longest_consecutive_length': longest_consec,
        'earliest_anomaly_timestamp': earliest_anom.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'latest_anomaly_timestamp': latest_anom.strftime('%Y-%m-%dT%H:%M:%SZ'),
        'max_value_in_window': max_value_in_window,
        'max_over_p95': max_over_p95,
        'sample_anomaly_timestamps': sample_ts
    })

# 5) Build result DataFrame, sort by severity (max_over_p95 desc), limit top 20
result_df_components = pd.DataFrame(rows)
if not result_df_components.empty:
    # Sort such that Inf (if any) come first
    result_df_components['_sort_key'] = result_df_components['max_over_p95'].replace(np.inf, np.finfo('float64').max)
    result_df_components = result_df_components.sort_values('_sort_key', ascending=False).drop(columns=['_sort_key'])
    cols = ['cmdb_id','kpi_name','p95','count_anomalous_points','longest_consecutive_length',
            'earliest_anomaly_timestamp','latest_anomaly_timestamp','max_value_in_window','max_over_p95','sample_anomaly_timestamps']
    result_df_components = result_df_components[cols].head(20)

# Display the compact result table
result_df_components

```
Out[4]:
```
```python
# Produce a plain-English summary based on the previously computed result_df_components in the IPython state.

if 'result_df_components' not in globals() or result_df_components is None or result_df_components.empty:
    summary = "No anomalies found for the specified components in the incident window."
else:
    df = result_df_components.copy()
    total_groups = len(df)
    earliest = df['earliest_anomaly_timestamp'].min()
    latest = df['latest_anomaly_timestamp'].max()
    # Top 3 by severity (max_over_p95)
    top3 = df.sort_values('max_over_p95', ascending=False).head(3)
    top_lines = []
    for _, r in top3.iterrows():
        top_lines.append(
            f"{r['cmdb_id']} / {r['kpi_name']} — p95={r['p95']:.6g}, max={r['max_value_in_window']:.6g}, "
            f"max_over_p95={r['max_over_p95']:.3f}, anomalies={r['count_anomalous_points']}, "
            f"longest_seq={r['longest_consecutive_length']}, {r['earliest_anomaly_timestamp']} to {r['latest_anomaly_timestamp']}"
        )
    # Common timing pattern
    most_common_start = df['earliest_anomaly_timestamp'].mode().iat[0] if not df['earliest_anomaly_timestamp'].empty else "N/A"
    # Build summary
    summary = (
        f"Analysis for components [ts-auth-service, ts-order-service, ts-route-service, ts-train-service, ts-travel-service]\n"
        f"Incident window: 2024-02-20 18:02:36 UTC to 2024-02-20 18:32:36 UTC\n\n"
        f"Total affected (groups with >=2 consecutive anomalous minutes): {total_groups}.\n"
        f"Overall anomaly time span among these groups: {earliest}  to  {latest}.\n\n"
        f"Top 3 most severe groups by max_over_p95:\n  1) {top_lines[0]}\n"
        + ("\n  2) " + top_lines[1] if len(top_lines) > 1 else "") \
        + ("\n  3) " + top_lines[2] if len(top_lines) > 2 else "") + "\n\n"
        f"Key observations:\n"
        f"- Most anomalies are short (longest_consecutive_length = 2 minutes) and cluster around {most_common_start} (many at 2024-02-20T18:05:00Z–18:06:00Z).\n"
        f"- Highest relative spikes observed:\n"
        f"  * ts-travel-service / latency-50 (largest max_over_p95 ≈ {df.sort_values('max_over_p95', ascending=False).iloc[0]['max_over_p95']:.3f})\n"
        f"  * ts-train-service / latency-50 (next most severe)\n"
        f"- Other notable signals: ts-travel-service (diskio, latency-90), ts-order-service (latency-50 and CPU), and socket/diskio anomalies occurring slightly later for some order/route/train groups.\n\n"
        f"Conclusion / recommendation:\n"
        f"- These five core services experienced brief, concurrent anomalies during the incident window, with ts-travel-service and ts-train-service showing the largest relative latency spikes.\n"
        f"- Investigate ts-travel-service and ts-train-service latency paths first, then check ts-order and ts-route for related I/O/socket spikes (some occurred at slightly different minutes).\n"
    )

summary
```

The original code execution output of IPython Kernel is also provided below for reference:

cmdb_id    kpi_name           p95  count_anomalous_points  longest_consecutive_length earliest_anomaly_timestamp latest_anomaly_timestamp  max_value_in_window  max_over_p95                     sample_anomaly_timestamps
11  ts-travel-service  latency-50  6.670439e-02                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         1.579622e-01      2.368092  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
9    ts-train-service  latency-50  9.223189e-03                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         2.173366e-02      2.356415  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
12  ts-travel-service  latency-90  5.616130e-01                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         9.947760e-01      1.771284  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
10  ts-travel-service      diskio  1.046606e+07                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         1.794862e+07      1.714935  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
3    ts-order-service  latency-50  1.323467e-02                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         2.032760e-02      1.535935  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
1    ts-order-service         cpu  9.594734e+00                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         1.439228e+01      1.500018  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
5    ts-route-service  latency-50  7.482443e-03                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         9.298657e-03      1.242730  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
0     ts-auth-service  latency-50  4.340968e-01                       2                           2       2024-02-20T18:05:00Z     2024-02-20T18:06:00Z         4.868382e-01      1.121497  [2024-02-20T18:05:00Z, 2024-02-20T18:06:00Z]
4    ts-order-service      socket  1.277000e+01                       2                           2       2024-02-20T18:19:00Z     2024-02-20T18:20:00Z         1.381667e+01      1.081963  [2024-02-20T18:19:00Z, 2024-02-20T18:20:00Z]
2    ts-order-service      diskio  6.450861e+04                       2                           2       2024-02-20T18:08:00Z     2024-02-20T18:09:00Z         6.711480e+04      1.040401  [2024-02-20T18:08:00Z, 2024-02-20T18:09:00Z]
7    ts-route-service      socket  2.418333e+01                       2                           2       2024-02-20T18:08:00Z     2024-02-20T18:09:00Z         2.495000e+01      1.031702  [2024-02-20T18:08:00Z, 2024-02-20T18:09:00Z]
8    ts-train-service      diskio  6.500449e+04                       2                           2       2024-02-20T18:08:00Z     2024-02-20T18:09:00Z         6.686305e+04      1.028591  [2024-02-20T18:08:00Z, 2024-02-20T18:09:00Z]
6    ts-route-service         mem  2.606671e+08                       2                           2       2024-02-20T18:10:00Z     2024-02-20T18:11:00Z         2.616100e+08      1.003617  [2024-02-20T18:10:00Z, 2024-02-20T18:11:00Z]
13  ts-travel-service         mem  2.853552e+08                       2                           2       2024-02-20T18:25:00Z     2024-02-20T18:26:00Z         2.863766e+08      1.003580  [2024-02-20T18:25:00Z, 2024-02-20T18:26:00Z]```
```